In [1]:
import pandas as pd
import statsmodels.api as sm

In [2]:
# Load dataset
df = pd.read_csv('FE-GWP1_model_selection_1.csv')

In [7]:
# Predictors
predictors = ["X1", "X2", "X3", "X4", "X5"]

In [8]:
def fit_model(data, variables):
    """
    Fit an OLS regression model using the specified predictors.
    """
    #X = sm.add_constant(data[variables])
    #return sm.OLS(y, X).fit()
    X = sm.add_constant(data[variables])
    y = data["Y"]

    return sm.OLS(y, X).fit()

In [9]:
def backward_adjusted_r2(data, predictors):
    """
    Backward elimination using adjusted R-squared.
    At each step, remove the variable whose removal produces
    the highest adjusted R-squared. Stop when removing a variable
    no longer improves adjusted R-squared.
    """
    selected = predictors.copy()
    history = []

    current_model = fit_model(data, selected)
    current_adj_r2 = current_model.rsquared_adj

    history.append({
        "step": 0,
        "removed": None,
        "variables": selected.copy(),
        "adjusted_R2": current_adj_r2
    })

    step = 1

    while len(selected) > 1:

        candidates = []

        for variable in selected:
            remaining = [x for x in selected if x != variable]
            model = fit_model(data, remaining)

            candidates.append({
                "removed": variable,
                "variables": remaining,
                "adjusted_R2": model.rsquared_adj
            })

        # Find the removal giving the highest adjusted R-squared
        best = max(candidates, key=lambda x: x["adjusted_R2"])

        if best["adjusted_R2"] > current_adj_r2:

            selected = best["variables"]
            current_adj_r2 = best["adjusted_R2"]

            history.append({
                "step": step,
                "removed": best["removed"],
                "variables": selected.copy(),
                "adjusted_R2": current_adj_r2
            })

            step += 1

        else:
            break

    final_model = fit_model(data, selected)

    return selected, final_model, history

In [10]:
backward_adjusted_r2(df, predictors)

(['X2', 'X3', 'X4', 'X5'],
 [{'step': 0,
   'removed': None,
   'variables': ['X1', 'X2', 'X3', 'X4', 'X5'],
   'adjusted_R2': np.float64(0.6301696768797365)},
  {'step': 1,
   'removed': 'X1',
   'variables': ['X2', 'X3', 'X4', 'X5'],
   'adjusted_R2': np.float64(0.6339742211445276)}])

In [11]:
def backward_aic(data, predictors):
    """
    Backward elimination using AIC.

    At each step, remove the variable whose removal produces
    the lowest AIC. Stop when no removal decreases AIC.
    """
    selected = predictors.copy()
    history = []

    current_model = fit_model(data, selected)
    current_aic = current_model.aic

    history.append({
        "step": 0,
        "removed": None,
        "variables": selected.copy(),
        "AIC": current_aic
    })

    step = 1

    while len(selected) > 1:

        candidates = []

        for variable in selected:
            remaining = [x for x in selected if x != variable]
            model = fit_model(data, remaining)

            candidates.append({
                "removed": variable,
                "variables": remaining,
                "AIC": model.aic
            })

        # Find the removal giving the lowest AIC
        best = min(candidates, key=lambda x: x["AIC"])

        if best["AIC"] < current_aic:

            selected = best["variables"]
            current_aic = best["AIC"]

            history.append({
                "step": step,
                "removed": best["removed"],
                "variables": selected.copy(),
                "AIC": current_aic
            })

            step += 1

        else:
            break

    final_model = fit_model(data, selected)

    return selected, final_model, history

In [12]:
backward_aic(df, predictors)

(['X2', 'X3', 'X4', 'X5'],
 [{'step': 0,
   'removed': None,
   'variables': ['X1', 'X2', 'X3', 'X4', 'X5'],
   'AIC': np.float64(262.5925282005836)},
  {'step': 1,
   'removed': 'X1',
   'variables': ['X2', 'X3', 'X4', 'X5'],
   'AIC': np.float64(260.61668419946966)}])

In [15]:
selected, final_model, history = backward_aic(
    df, predictors
)

print(final_model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.649
Model:                            OLS   Adj. R-squared:                  0.634
Method:                 Least Squares   F-statistic:                     43.87
Date:                Fri, 18 Sep 2026   Prob (F-statistic):           8.29e-21
Time:                        12:12:21   Log-Likelihood:                -125.31
No. Observations:                 100   AIC:                             260.6
Df Residuals:                      95   BIC:                             273.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.1893      0.089     13.333      0.0

In [13]:
def backward_bic(data, predictors):
    """
    Backward elimination using BIC.

    At each step, remove the variable whose removal produces
    the lowest BIC. Stop when no removal decreases BIC.
    """
    selected = predictors.copy()
    history = []

    current_model = fit_model(data, selected)
    current_bic = current_model.bic

    history.append({
        "step": 0,
        "removed": None,
        "variables": selected.copy(),
        "BIC": current_bic
    })

    step = 1

    while len(selected) > 1:

        candidates = []

        for variable in selected:
            remaining = [x for x in selected if x != variable]
            model = fit_model(data, remaining)

            candidates.append({
                "removed": variable,
                "variables": remaining,
                "BIC": model.bic
            })

        # Find the removal giving the lowest BIC
        best = min(candidates, key=lambda x: x["BIC"])

        if best["BIC"] < current_bic:

            selected = best["variables"]
            current_bic = best["BIC"]

            history.append({
                "step": step,
                "removed": best["removed"],
                "variables": selected.copy(),
                "BIC": current_bic
            })

            step += 1

        else:
            break

    final_model = fit_model(data, selected)

    return selected, final_model, history

In [14]:
backward_bic(df, predictors)

(['X2', 'X3', 'X4', 'X5'],
 [{'step': 0,
   'removed': None,
   'variables': ['X1', 'X2', 'X3', 'X4', 'X5'],
   'BIC': np.float64(278.22354931651216)},
  {'step': 1,
   'removed': 'X1',
   'variables': ['X2', 'X3', 'X4', 'X5'],
   'BIC': np.float64(273.64253512941013)}])

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = df[["X1", "X2", "X3", "X4", "X5"]]
y = df["Y"]

lasso = make_pipeline(
    StandardScaler(),
    LassoCV(cv=10, random_state=42)
)

lasso.fit(X, y)

model = lasso.named_steps["lassocv"]

print("Best alpha:", model.alpha_)

for variable, coefficient in zip(X.columns, model.coef_):
    print(variable, coefficient)